In [ ]:
# SEQC

# !pip -q install transformers datasets evaluate scikit-learn pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Libraries loaded for SEQC approach.")

In [ ]:
# load dataset and prepare input text + labels

import pandas as pd

train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def build_seqc_input(row):
    src = safe_text(row["text_src"])
    tgt = safe_text(row["text_tgt"])

    if src and tgt:
        return src + " [SEP] " + tgt
    elif src:
        return src
    else:
        return tgt

train_df["input_text"] = train_df.apply(build_seqc_input, axis=1)
val_df["input_text"] = val_df.apply(build_seqc_input, axis=1)
test_df["input_text"] = test_df.apply(build_seqc_input, axis=1)

labels = train_df["label"].unique()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"] = val_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nLabel mapping:")
print(label2id)

print("\nSample input:")
print(train_df["input_text"].iloc[0])

print("\nSample label:")
print(train_df["label"].iloc[0], "->", train_df["label_id"].iloc[0])

In [ ]:
# convert pandas dataframes to Hugging Face datasets

from datasets import Dataset

train_seqc = Dataset.from_pandas(train_df[["input_text", "label_id"]])
val_seqc = Dataset.from_pandas(val_df[["input_text", "label_id"]])
test_seqc = Dataset.from_pandas(test_df[["input_text", "label_id"]])

print("Train dataset:", train_seqc)
print("Validation dataset:", val_seqc)
print("Test dataset:", test_seqc)

In [ ]:
#  load tokenizer and sequence classification model

from transformers import AutoTokenizer, AutoModelForSequenceClassification

seqc_model_name = "distilbert-base-uncased"

seqc_tokenizer = AutoTokenizer.from_pretrained(seqc_model_name)

seqc_model = AutoModelForSequenceClassification.from_pretrained(
    seqc_model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

print("SEQC model loaded successfully:", seqc_model_name)
print("Number of labels:", len(label2id))

In [ ]:
#  tokenize the datasets

max_length = 128

def tokenize_seqc(example):
    return seqc_tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

train_seqc_tokenized = train_seqc.map(tokenize_seqc, batched=True)
val_seqc_tokenized = val_seqc.map(tokenize_seqc, batched=True)
test_seqc_tokenized = test_seqc.map(tokenize_seqc, batched=True)

print("SEQC tokenization completed.")
print(train_seqc_tokenized[0])

In [ ]:
# set format and rename labels

train_seqc_tokenized = train_seqc_tokenized.rename_column("label_id", "labels")
val_seqc_tokenized = val_seqc_tokenized.rename_column("label_id", "labels")
test_seqc_tokenized = test_seqc_tokenized.rename_column("label_id", "labels")

columns = ["input_ids", "attention_mask", "labels"]

train_seqc_tokenized.set_format(type="torch", columns=columns)
val_seqc_tokenized.set_format(type="torch", columns=columns)
test_seqc_tokenized.set_format(type="torch", columns=columns)

print("SEQC dataset ready for training.")
print(train_seqc_tokenized[0])

In [ ]:
# define metrics

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_seqc_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("SEQC metrics function ready.")

In [ ]:
# train the SEQC model

import torch
from transformers import TrainingArguments, Trainer

seqc_training_args = TrainingArguments(
    output_dir="./seqc_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

seqc_trainer = Trainer(
    model=seqc_model,
    args=seqc_training_args,
    train_dataset=train_seqc_tokenized,
    eval_dataset=val_seqc_tokenized,
    compute_metrics=compute_seqc_metrics
)

seqc_trainer.train()

In [ ]:
# evaluate on test set

seqc_test_results = seqc_trainer.evaluate(test_seqc_tokenized)
print(seqc_test_results)

In [ ]:
#  get predictions on test set

seqc_pred_output = seqc_trainer.predict(test_seqc_tokenized)

print("Prediction step completed.")
print("Prediction shape:", seqc_pred_output.predictions.shape)

In [ ]:
#  convert prediction scores to class labels

import numpy as np

seqc_predictions = np.argmax(seqc_pred_output.predictions, axis=1)
seqc_true_labels = seqc_pred_output.label_ids

seqc_pred_names = [id2label[i] for i in seqc_predictions]
seqc_true_names = [id2label[i] for i in seqc_true_labels]

print("Sample predictions:")
for i in range(5):
    print("Pred:", seqc_pred_names[i], "| True:", seqc_true_names[i])

In [ ]:
#  classification report

from sklearn.metrics import classification_report

print(classification_report(seqc_true_names, seqc_pred_names, labels=list(label2id.keys()), zero_division=0))

In [ ]:
#  confusion matrix

from sklearn.metrics import confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

seqc_label_list = list(label2id.keys())

seqc_cm = confusion_matrix(seqc_true_names, seqc_pred_names, labels=seqc_label_list)

seqc_cm_df = pd.DataFrame(seqc_cm, index=seqc_label_list, columns=seqc_label_list)
print(seqc_cm_df)

plt.figure(figsize=(8, 6))
plt.imshow(seqc_cm, interpolation="nearest")
plt.title("SEQC Confusion Matrix")
plt.colorbar()
plt.xticks(range(len(seqc_label_list)), seqc_label_list, rotation=45)
plt.yticks(range(len(seqc_label_list)), seqc_label_list)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
#  save SEQC model to Google Drive

from google.colab import drive
drive.mount('/content/drive')

seqc_save_path = "/content/drive/MyDrive/seqc_final_model"

seqc_trainer.save_model(seqc_save_path)
seqc_tokenizer.save_pretrained(seqc_save_path)

print("SEQC model saved at:", seqc_save_path)